# Raw 20 ms SBP trajectories around phoneme-bigram transitions

This notebook uses transcript phonemes for bigram identity and the frozen step-18,300 GRU only for reference-constrained CTC timing. The plotted trajectories are unsmoothed, clipped-FP16 SBP from the first 128 area-6v channels at 20 ms sampling. GRU timing remains native to an 80 ms stride, so these are **GRU-aligned raw-neural trajectories**, not independently timed articulatory movements.

The analysis is descriptive and transductive: all 24 `competition_test` sessions select the 66 bigrams, estimate session-wise nuisance statistics, fit PCA, and contribute to rankings. Run cells 1 through 8 in order. Run 9 only after saving and inspecting the outputs.

In [ ]:
# RUN 1 — Mount Drive, update a clean repository checkout, and install missing lightweight packages.
from google.colab import drive
drive.mount('/content/drive')

import importlib.util
import os
from pathlib import Path
import subprocess
import sys

REPO_DIR = Path('/content/utah-ssl')
REPO_URL = 'https://github.com/ethan-read/utah-ssl.git'
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    dirty = subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO_DIR, text=True).strip()
    if dirty:
        raise RuntimeError(f'Repository checkout has local changes; preserve or remove them first:\n{dirty}')
    subprocess.run(['git', 'fetch', 'origin'], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', 'main'], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', 'main'], cwd=REPO_DIR, check=True)
required = {'numpy': 'numpy', 'pandas': 'pandas', 'matplotlib': 'matplotlib', 'sklearn': 'scikit-learn'}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
GIT_COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
print('Repository:', REPO_DIR)
print('Commit:', GIT_COMMIT)

## Configuration

Run the next cell after deciding between the engineering smoke test and the complete 880-example analysis. Smoke mode deliberately relaxes the frequency/count assertions and writes to a separate directory; it is not scientific evidence.

In [ ]:
# RUN 2 — Declare the source artifacts, full/smoke configuration, and final write destination.
from experiments.manifolds import BigramTrajectoryConfig

UTAH_SSL_ROOT = Path('/content/drive/MyDrive/utah_ssl')
MODEL_DIR = UTAH_SSL_ROOT / 'data/representations/willett_manifolds/gru_layerwise_b2t24_step18300_v1/gru_best_step18300_all_val_sessions'
RAW_SBP_CACHE_ROOT = UTAH_SSL_ROOT / 'data/cache_v1_sbpclip12500_fp16_raw'
OUTPUT_NAME = 'raw_20ms_sbp_bigram_transition_pca_t12_all24_v1'
SMOKE = False
OVERWRITE = False

if SMOKE:
    CONFIG = BigramTrajectoryConfig(minimum_transcript_count=2, bootstrap_repetitions=25, expected_examples=None, expected_sessions=None, expected_bigram_count=None, max_examples_per_session=3, smoke=True)
    OUTPUT_DIR = UTAH_SSL_ROOT / 'outputs/neural_trajectories' / f'{OUTPUT_NAME}_smoke'
else:
    CONFIG = BigramTrajectoryConfig()
    OUTPUT_DIR = UTAH_SSL_ROOT / 'outputs/neural_trajectories' / OUTPUT_NAME

if OUTPUT_DIR.exists() and not OVERWRITE:
    raise FileExistsError(f'Output already exists: {OUTPUT_DIR}. Inspect it or explicitly set OVERWRITE=True.')
print('Mode:', 'SMOKE (engineering only)' if SMOKE else 'FULL')
print('GRU timing export:', MODEL_DIR)
print('Raw SBP cache:', RAW_SBP_CACHE_ROOT)
print('Final write destination:', OUTPUT_DIR)

## Validate and join sources

The next cell loads only the compact logits and tables—not the five hidden-state matrices. It validates checkpoint step, patching, vocabulary, cache identity, clipped-FP16 raw SBP, 128-channel signal selection, split membership, and the exact example-ID/session join.

In [ ]:
# RUN 3 — Validate the GRU export and canonical raw-SBP cache contracts.
from experiments.manifolds import prepare_bigram_sources

sources = prepare_bigram_sources(MODEL_DIR, RAW_SBP_CACHE_ROOT, config=CONFIG)
if not SMOKE:
    assert len(sources.selected_examples) == 880
    assert sources.selected_examples.session_id.nunique() == 24
    assert len(sources.candidate_pairs) == 66
print('Joined examples:', len(sources.selected_examples))
print('Sessions:', sources.selected_examples.session_id.nunique())
print('SignalSpec:', sources.signal_spec.to_dict())
print('Candidate bigrams:', len(sources.candidate_pairs))

In [ ]:
# RUN 4 — Inspect transcript counts and the exact selected bigram identities before reading raw arrays.
import pandas as pd

candidate_rows = []
for first_id, second_id in sources.candidate_pairs:
    candidate_rows.append({'first_id': first_id, 'second_id': second_id, 'bigram': f'{sources.symbol_by_id[first_id]}-{sources.symbol_by_id[second_id]}', 'transcript_count': sources.transcript_counts[(first_id, second_id)]})
candidate_table = pd.DataFrame(candidate_rows).sort_values('transcript_count', ascending=False).reset_index(drop=True)
if not SMOKE:
    assert len(candidate_table) == 66
    assert candidate_table.transcript_count.min() >= 50
display(candidate_table)

## Extract transition windows

This is the main Drive-read step. It checks every cache reference against the exported reference, accumulates session statistics from every selected utterance bin, performs hard CTC alignment, and extracts nominal and ±40 ms windows without truncation.

In [ ]:
# RUN 5 — Stream raw SBP once, compute session moments, align transitions, and extract 29-bin paths.
from experiments.manifolds import build_bigram_event_set

event_set = build_bigram_event_set(MODEL_DIR, RAW_SBP_CACHE_ROOT, config=CONFIG, progress=print, sources=sources)
if not SMOKE:
    assert len(event_set.counts) == 66
    assert event_set.events.session_id.nunique() == 24
    assert event_set.paths_by_jitter[0].shape[1:] == (29, 128)
display(event_set.counts.sort_values('jitter_+0_valid_count', ascending=False))
display(event_set.diagnostics.groupby('status').agg(examples=('example_id', 'count'), candidate_occurrences=('candidate_occurrences', 'sum'), valid_occurrences=('nominal_valid_occurrences', 'sum')).reset_index())
display(event_set.exclusions.groupby('reason', dropna=False).size().rename('count').reset_index())

## Fit PCA and rank trajectories

The primary PCA removes each event's temporal mean; both PCA fits first average events within each bigram/session and then weight sessions equally within each bigram. The state control retains absolute session-normalized position. Rankings use the same session-equal mean paths. `trajectory_captured_fraction` is not PCA explained variance.

In [ ]:
# RUN 6 — Fit change/state PCA, compute session-equal rankings, bootstrap intervals, and jitter sensitivity.
import numpy as np
from experiments.manifolds import analyze_bigram_event_set

result = analyze_bigram_event_set(event_set, config=CONFIG)
for pca in (result.change_pca, result.state_pca):
    assert np.isfinite(pca.eigenvalues).all()
    assert pca.eigenvalues.min() >= 0
    assert np.all(np.diff(np.cumsum(pca.explained_variance_ratio)) >= -1e-12)
if not SMOKE:
    assert len(result.ranking) == 66
display(result.pca_variance.query('component <= 12'))
display(result.ranking.drop(columns=[column for column in result.ranking if column.startswith('k')]).head(20))

## Inspect figures

The ranking is primary. Interpret it together with magnitude, session coverage, alignment confidence, and leave-one-session-out correlation. The change grids show within-window movement; the state grids show the uncentered control. Shared axes permit visual scale comparisons.

In [ ]:
# RUN 7 — Render the scree, ranking, and all paginated change/state trajectory grids.
import matplotlib.pyplot as plt
from experiments.manifolds import make_bigram_trajectory_figures

preview_figures = make_bigram_trajectory_figures(result)
for name, figure in preview_figures.items():
    print(name)
    display(figure)
    plt.close(figure)

## Save and validate

Run this only after inspecting the tables and figures. The writer stages every artifact, reopens all required tables, arrays, metadata, and figures, and atomically promotes the completed directory.

In [ ]:
# RUN 8 — Save atomically, reopen required artifacts, and print the compact summary.
import json
from experiments.manifolds import save_bigram_trajectory_result

saved = save_bigram_trajectory_result(result, OUTPUT_DIR, git_commit=GIT_COMMIT, overwrite=OVERWRITE)
summary = json.loads((OUTPUT_DIR / 'summary.json').read_text())
marker = json.loads((OUTPUT_DIR / '_SUCCESS.json').read_text())
assert marker['status'] == 'complete'
assert summary['status'] == ('smoke_engineering_only' if SMOKE else 'complete')
print('Artifact reopen checks passed.')
print('Saved:', saved['output_dir'])
print(json.dumps(summary, indent=2))

## Teardown

Run this final cell only after Run 8 completes and the saved directory has been inspected.

In [ ]:
# RUN 9 — Flush Google Drive, unmount it, and release the Colab runtime.
from google.colab import drive, runtime
drive.flush_and_unmount()
runtime.unassign()